In [ ]:
# Installation
!pip install opencv-python-headless tqdm Pillow torchvision

In [ ]:
import os
import cv2
import glob
import shutil
import numpy as np
from tqdm import tqdm
from PIL import Image
from torchvision import transforms

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive1')


Mounted at /content/drive1


In [ ]:
import os
import cv2
import glob
import shutil
import numpy as np
from tqdm import tqdm
from PIL import Image
from torchvision import transforms

# ----- AUGMENTATION SETUP -----

def get_color_jitter_transform():
    """Returns a torchvision color jitter transform."""
    return transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    )

# ----- DIRECTORY HELPERS -----

def validate_output_dir(shared_dir):
    """Validates that the shared_dir exists and is writable."""
    if not os.path.exists(shared_dir):
        raise ValueError(f"Target directory {shared_dir} does not exist.")
    if not os.access(shared_dir, os.W_OK):
        raise ValueError(f"No write permission for {shared_dir}.")

def prepare_class_dirs(data_dir, shared_dir):
    """Creates matching class directories in shared_dir."""
    class_names = sorted(os.listdir(data_dir))
    for cls in class_names:
        class_path = os.path.join(shared_dir, cls)
        os.makedirs(class_path, exist_ok=True)
    return class_names

def clear_existing_videos(folder):
    """Removes all .mp4, .avi, .MOV videos from a directory."""
    extensions = ("*.mp4", "*.avi", "*.MOV")
    for ext in extensions:
        for file in glob.glob(os.path.join(folder, ext)):
            os.remove(file)

def copy_original_videos(src_dir, dest_dir):
    """Copies all videos from source to destination directory."""
    video_paths = glob.glob(os.path.join(src_dir, "*.mp4")) + \
                  glob.glob(os.path.join(src_dir, "*.avi")) + \
                  glob.glob(os.path.join(src_dir, "*.MOV"))

    for video_path in video_paths:
        shutil.copy(video_path, os.path.join(dest_dir, os.path.basename(video_path)))
    return video_paths

# ----- VIDEO PROCESSING HELPERS -----

def load_video_frames(video_path):
    """Loads all frames from a video and returns frame list and metadata."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Failed to open video: {video_path}")
        return [], 0, 0, 0, 0

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(rgb_frame)

    cap.release()
    return frames, frame_count, fps, width, height

def save_augmented_video(frames, path, fps, width, height):
    """Saves frames to video using MPEG-4 codec."""
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(path, fourcc, fps, (width, height))
    for frame in frames:
        bgr_frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        out.write(bgr_frame)
    out.release()

# ----- AUGMENTATION TYPES -----

def apply_color_jitter(frames, transform):
    """Applies color jitter to each frame."""
    return [
        cv2.cvtColor(np.array(transform(Image.fromarray(f))), cv2.COLOR_RGB2BGR)
        for f in frames
    ]

# tilt + (RGB -> BGR)
def apply_rotation(frames, angle, width, height):
    """Applies affine rotation to all frames."""
    matrix = cv2.getRotationMatrix2D((width // 2, height // 2), angle, 1.0)
    return [cv2.warpAffine(f, matrix, (width, height)) for f in frames]

def apply_slow_motion(frames, factor):
    """Repeats frames to simulate slow motion."""
    return [frame for frame in frames for _ in range(int(factor))]

def apply_fast_motion(frames, factor):
    """Skips frames to simulate fast motion."""
    indices = np.linspace(0, len(frames) - 1, int(len(frames) / factor)).astype(int)
    return [frames[i] for i in indices]

# ----- MAIN FUNCTION -----

def create_augmented_dataset_to_shared(data_dir, shared_dir, num_augmented_per_type=10):
    """
    Copies original videos and creates 5 types of augmentations per class
    (slow, fast, color jitter, rotate left, rotate right).
    Saves all in the shared_dir with .mp4 format.
    """
    validate_output_dir(shared_dir)
    print(f"Using output directory: {shared_dir}")

    color_jitter = get_color_jitter_transform()
    class_names = prepare_class_dirs(data_dir, shared_dir)

    for cls_name in tqdm(class_names, desc="Processing Classes"):
        class_input_dir = os.path.join(data_dir, cls_name)
        class_output_dir = os.path.join(shared_dir, cls_name)

        if not os.path.isdir(class_input_dir):
            print(f"Skipping non-directory: {class_input_dir}")
            continue

        clear_existing_videos(class_output_dir)
        video_paths = copy_original_videos(class_input_dir, class_output_dir)

        for i, video_path in enumerate(video_paths[:50]):
            aug_type = i // num_augmented_per_type
            base_idx = i % num_augmented_per_type
            aug_name = ["slow", "fast", "color", "left", "right"][aug_type]
            output_name = f"{aug_name}_video{base_idx+1}.mp4"
            output_path = os.path.join(class_output_dir, output_name)

            frames, frame_count, fps, width, height = load_video_frames(video_path)
            if not frames:
                continue

            # Apply augmentation
            if aug_type == 0:
                new_fps = fps / 1.2
                aug_frames = apply_slow_motion(frames, 1.2)
            elif aug_type == 1:
                new_fps = fps
                aug_frames = apply_fast_motion(frames, 1.5)
            elif aug_type == 2:
                new_fps = fps
                aug_frames = apply_color_jitter(frames, color_jitter)
            elif aug_type == 3:
                new_fps = fps
                aug_frames = apply_rotation(frames, -6, width, height)
            elif aug_type == 4:
                new_fps = fps
                aug_frames = apply_rotation(frames, 6, width, height)
            else:
                continue

            save_augmented_video(aug_frames, output_path, new_fps, width, height)

            # Log results
            cap = cv2.VideoCapture(output_path)
            out_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else 0
            out_fps = cap.get(cv2.CAP_PROP_FPS) or new_fps
            cap.release()

            print(f"Saved {output_path}: {out_frame_count} frames, {out_fps:.2f} FPS, duration {out_frame_count / out_fps:.2f}s")

            expected = {
                0: int(frame_count * 1.2),
                1: int(frame_count / 1.5)
            }.get(aug_type, frame_count)

            if out_frame_count != expected:
                print(f"Frame mismatch: expected {expected}, got {out_frame_count}")

# Example usage
if __name__ == "__main__":
    data_dir = "/content/drive1/MyDrive/Dataset(KSL) - Merged"
    shared_dir = "/content/drive1/MyDrive/major_project/dataset/mixture2"
    create_augmented_dataset_to_shared(data_dir, shared_dir)
    print(f"All videos (original + augmented) saved in {shared_dir}")


Using target directory: /content/drive1/MyDrive/major_project/dataset/mixture2
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Afternoon
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Apple
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/April
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/August
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Banana
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Day
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/December
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Evening
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Febraury
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Friday
Created directory: /content/drive1/MyDrive/major_project/dataset/mixture2/Grapes
Created directory: /content

Copying originals for Afternoon:   0%|          | 0/70 [00:00<?, ?it/s]

Copied /content/drive1/MyDrive/Dataset(KSL) - Merged/Afternoon/20241107_134140.mp4 to /content/drive1/MyDrive/major_project/dataset/mixture2/Afternoon/20241107_134140.mp4
Copied /content/drive1/MyDrive/Dataset(KSL) - Merged/Afternoon/20241107_134202.mp4 to /content/drive1/MyDrive/major_project/dataset/mixture2/Afternoon/20241107_134202.mp4



Copying originals for Afternoon:   4%|▍         | 3/70 [00:00<00:04, 16.28it/s]

Copied /content/drive1/MyDrive/Dataset(KSL) - Merged/Afternoon/20241107_140048.mp4 to /content/drive1/MyDrive/major_project/dataset/mixture2/Afternoon/20241107_140048.mp4
Copied /content/drive1/MyDrive/Dataset(KSL) - Merged/Afternoon/20241107_140056.mp4 to /content/drive1/MyDrive/major_project/dataset/mixture2/Afternoon/20241107_140056.mp4


...

Saved /content/drive1/MyDrive/major_project/dataset/mixture2/Winter/right_video8.mp4: 47 frames, 30.64 FPS, duration 1.53s
Processing /content/drive1/MyDrive/Dataset(KSL) - Merged/Winter/win_19b.mp4: 167 frames, 60.04002668445631 FPS, 720x1280 resolution



Augmenting for Winter:  98%|█████████▊| 49/50 [03:18<00:02,  2.58s/it]

Saved /content/drive1/MyDrive/major_project/dataset/mixture2/Winter/right_video9.mp4: 167 frames, 60.04 FPS, duration 2.78s
Processing /content/drive1/MyDrive/Dataset(KSL) - Merged/Winter/win_19a.mp4: 131 frames, 60.04002668445631 FPS, 720x1280 resolution



Processing classes: 100%|██████████| 33/33 [2:34:16<00:00, 280.51s/it]

Saved /content/drive1/MyDrive/major_project/dataset/mixture2/Winter/right_video10.mp4: 131 frames, 60.04 FPS, duration 2.18s
All videos (original and augmented) saved in /content/drive1/MyDrive/major_project/dataset/mixture2
